# checking JEPA with linear classifyer

In [ ]:
from net2brain.utils.download_datasets import DatasetBonnerPNAS2017, DatasetAlgonauts_NSD_Shared
from pprint import pprint

#paths_bonner = DatasetBonnerPNAS2017.load_dataset()
#pprint(paths_bonner)

paths_NSD_shared  = DatasetAlgonauts_NSD_Shared.load_dataset()
pprint(paths_NSD_shared)
paths_NSD_shared.keys()
import os



import os

stimuli_path = paths_NSD_shared["images"]

# count files
num_images = len([f for f in os.listdir(stimuli_path) if f.endswith(('.jpg', '.png'))])

#print("Number of images:", num_images)

Run one image through the model and print the output to test shape:

In [ ]:
import os
from PIL import Image
import torch
from torchvision import transforms

# load one image

# pick first image
img_name = os.listdir(stimuli_path)[0]
img_path = os.path.join(stimuli_path, img_name)

print("Using image:", img_path)

img = Image.open(img_path).convert("RGB")



# basic preprocessing (adjust if you already have one)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

x = transform(img).unsqueeze(0)  # shape: (1, 3, 224, 224)

# forward pass
with torch.no_grad():
    out = vit_model(x)

# inspect
print(type(out))

if isinstance(out, (tuple, list)):
    for i, o in enumerate(out):
        print(f"Output {i} shape:", o.shape)
elif isinstance(out, dict):
    for k, v in out.items():
        print(k, v.shape)
else:
    print("Output shape:", out.shape)

Testing shapes of dif embedings and make sure they are different

In [ ]:
with torch.no_grad():
    outputs = vit_model(x)

last_hidden = outputs.last_hidden_state  # (1, 257, 1280)

# 1. Mean pooling (no CLS)
mean_pool = last_hidden[:, 1:, :].mean(dim=1)

# 2. CLS token
cls_token = last_hidden[:, 0, :]

# 3. Pooler output
pooler = outputs.pooler_output

print("Mean pool shape:", mean_pool.shape)
print("CLS shape:", cls_token.shape)
print("Pooler shape:", pooler.shape)

# check if vectors are different
print("\nDifferences:")
print("mean vs cls:", torch.norm(mean_pool - cls_token))
print("mean vs pooler:", torch.norm(mean_pool - pooler))
print("cls vs pooler:", torch.norm(cls_token - pooler))

embedding extractor function. lets you switch between: "mean" "cls" "pooler" to know which layer is best at classifying

In [ ]:
import torch.nn.functional as F

def extract_embedding(model, x, mode="mean"):
    with torch.no_grad():
        outputs = model(x) # this is the JEPA forward pass

    last_hidden = outputs.last_hidden_state

    if mode == "mean":
        emb = last_hidden[:, 1:, :].mean(dim=1)

    elif mode == "cls":
        emb = last_hidden[:, 0, :]

    elif mode == "pooler":
        emb = outputs.pooler_output

    else:
        raise ValueError("Unknown mode")

    # normalize (important for linear probing stability)
    emb = F.normalize(emb, dim=-1)

    return emb

In [ ]:
# test it
emb = extract_embedding(vit_model, x, mode="mean")
print(emb.shape)
print(torch.norm(emb))  # should be ~1

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os

class ImageNet10EmbeddingDataset(Dataset):
    def __init__(self, root_dir, model, transform, mode="mean", device="cuda"):
        self.root_dir = root_dir
        self.model = model
        self.transform = transform
        self.mode = mode
        self.device = device

        self.samples = []
        self.class_to_idx = {}

        # build dataset
        for i, class_name in enumerate(sorted(os.listdir(root_dir))):
            class_path = os.path.join(root_dir, class_name)

            if not os.path.isdir(class_path):
                continue

            self.class_to_idx[class_name] = i

            for fname in os.listdir(class_path):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((
                        os.path.join(class_path, fname),
                        i
                    ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        img = Image.open(img_path).convert("RGB")
        x = self.transform(img).unsqueeze(0).to(self.device)

        emb = extract_embedding(self.model, x, mode=self.mode)

        return emb.squeeze(0), label

In [ ]:
#test it
dataset = ImageNet10EmbeddingDataset(
    root_dir= r"C:\Users\louai\Net2Brain\net2brain\imagenette2\imagenette2\train",
    model=vit_model,
    transform=transform,
    mode="mean",
    device=device
)

x, y = dataset[0]

print(x.shape)  # should be (1280,)
print(y)        # int label

In [ ]:
import torch
from tqdm import tqdm

def precompute_embeddings(dataset):
    X = []
    y = []

    for i in tqdm(range(len(dataset))):
        emb, label = dataset[i]  # this calls __getitem__ in ImageNet10EmbeddingDataset which calls extract_embedding(model=jepa, x=imgs) 
        X.append(emb.cpu())
        y.append(label)

    X = torch.stack(X)   # (N, 1280)
    y = torch.tensor(y)  # (N,)

    return X, y

In [ ]:
# only run once
X_train, y_train = precompute_embeddings(dataset)

print(X_train.shape)
print(y_train.shape)

#save the embeddings
torch.save(X_train, "X_train.pt")
torch.save(y_train, "y_train.pt")

In [ ]:
# reload them
import torch

X_train = torch.load("X_train.pt")
y_train = torch.load("y_train.pt")

print(X_train.shape)
print(y_train.shape)

Define the linear classifier

In [ ]:
import torch.nn as nn

class LinearProbe(nn.Module):
    def __init__(self, in_dim=1280, num_classes=10):
        super().__init__()
        self.linear = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

In [ ]:
# test it
model = LinearProbe().to(device)

# fake batch
x = torch.randn(32, 1280).to(device)

out = model(x)

print(out.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

In [ ]:
for xb, yb in train_loader:
    print(xb.shape, yb.shape)
    break

In [ ]:
import torch.optim as optim

model = LinearProbe().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
#train
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        # forward
        logits = model(xb)
        loss = criterion(logits, yb)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [ ]:
def compute_accuracy(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == yb).sum().item()
            total += yb.size(0)

    return correct / total

In [ ]:
train_acc = compute_accuracy(model, train_loader, device)
print("Train accuracy:", train_acc)
#High accuracy (~80–100%) → embeddings are very separable

In [ ]:
#validate
val_root = r"C:\Users\louai\Net2Brain\net2brain\imagenette2\imagenette2\val"
val_dataset = ImageNet10EmbeddingDataset(
    root_dir=val_root,
    model=vit_model,
    transform=transform,
    mode="mean",
    device=device
)

In [ ]:
X_val, y_val = precompute_embeddings(val_dataset)

print(X_val.shape, y_val.shape)

In [ ]:
val_dataset = TensorDataset(X_val.to(device), y_val.to(device))

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

In [ ]:
val_acc = compute_accuracy(model, val_loader, device)
print("Validation accuracy:", val_acc)